---
## 7. Config A — Hyperparameter Optimisation

The model-free agents in Sections 3 and 4 used default hyperparameters (alpha=0.3, 50k episodes). Looking at the learning curves, both were still improving at the end of training — they had not fully converged.

We run a focused optimisation with two goals:
1. **Alpha grid search** for both SARSA and Q-Learning (5 values × 2 algorithms at 200k episodes). This is the most analytically valuable experiment for the report.
2. **Final optimised run** at 200k episodes using the best alpha found for each algorithm, including a **stability analysis** — because in a clinical setting, a consistent policy matters as much as a high average survival rate.

Both algorithms are tuned under identical conditions so the comparison is fair.

> **Note on Policy Iteration as the ceiling:** PI finds the optimal policy given the reward function and transition matrix P. However, P is estimated from finite MIMIC-III data, so PI optimises over a noisy model. The gap between PI and model-free agents therefore represents both the cost of not having the model AND the noise in P itself. PI is the right benchmark for Config A comparison but is not an absolute upper bound on clinical performance.


### 7.1 — Alpha Grid Search: SARSA and Q-Learning

The learning rate alpha controls how aggressively the agent updates its Q-values after each experience. Too small and learning is too slow to converge within our episode budget. Too large and Q-values become unstable.

We test alpha ∈ {0.05, 0.1, 0.2, 0.3, 0.5} for both algorithms at 200k episodes each.


In [ ]:
# ── Alpha grid search: both algorithms, 200k episodes ────────────────────────
# 10 training runs total (5 alphas × 2 algorithms)
# Estimated runtime: ~5-10 minutes depending on hardware

alpha_values = [0.05, 0.1, 0.2, 0.3, 0.5]
N_OPT_EPISODES = 200_000

alpha_grid_results = {}  # keyed by (algo_name, alpha)
alpha_grid_returns = {}

for algo_name, algo_fn in [('SARSA', sarsa), ('Q-Learning', q_learning)]:
    print(f'\n── {algo_name} ──')
    for alpha_val in alpha_values:
        print(f'  alpha={alpha_val} | {N_OPT_EPISODES:,} episodes...', end=' ', flush=True)
        Q_tmp, returns_tmp = algo_fn(
            n_episodes=N_OPT_EPISODES,
            alpha=alpha_val,
            gamma=GAMMA,
            epsilon_start=1.0,
            epsilon_min=0.01,
            seed=SEED,
        )
        policy_tmp  = np.argmax(Q_tmp, axis=1)
        results_tmp = evaluate_policy_tabular(policy_tmp)
        alpha_grid_results[(algo_name, alpha_val)] = results_tmp
        alpha_grid_returns[(algo_name, alpha_val)] = returns_tmp
        print(f'survival={results_tmp["survival_rate"]:.1f}%  '
              f'return={results_tmp["mean_return"]:.4f}')

# Identify best alpha per algorithm
best_alpha_sarsa = max(alpha_values,
    key=lambda a: alpha_grid_results[('SARSA', a)]['survival_rate'])
best_alpha_ql    = max(alpha_values,
    key=lambda a: alpha_grid_results[('Q-Learning', a)]['survival_rate'])

print(f'\nBest alpha — SARSA      : {best_alpha_sarsa} '
      f'({alpha_grid_results[("SARSA", best_alpha_sarsa)]["survival_rate"]:.1f}%)')
print(f'Best alpha — Q-Learning : {best_alpha_ql} '
      f'({alpha_grid_results[("Q-Learning", best_alpha_ql)]["survival_rate"]:.1f}%)')


In [ ]:
# ── Plot: alpha grid search for both algorithms ───────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (algo, color, best_a) in enumerate([
    ('SARSA',      'steelblue',  best_alpha_sarsa),
    ('Q-Learning', 'darkorange', best_alpha_ql),
]):
    survival_by_alpha = [alpha_grid_results[(algo, a)]['survival_rate']
                         for a in alpha_values]

    # Left: bar chart survival vs alpha
    light_color = '#aed6f1' if algo == 'SARSA' else '#fad7a0'
    bar_colors  = [color if a == best_a else light_color for a in alpha_values]
    bars = axes[row, 0].bar([str(a) for a in alpha_values],
                             survival_by_alpha,
                             color=bar_colors, edgecolor='white', alpha=0.9)
    axes[row, 0].axhline(pi_results['survival_rate'], color='tomato',
                         linestyle='--', linewidth=1.5,
                         label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
    axes[row, 0].axhline(survival_rate, color='gray', linestyle=':',
                         linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
    axes[row, 0].set_xlabel('Learning rate (alpha)')
    axes[row, 0].set_ylabel('Survival Rate (%)')
    axes[row, 0].set_title(
        f'{algo}: Survival Rate vs Alpha\n({N_OPT_EPISODES//1000}k episodes, highlighted = best)',
        fontweight='bold')
    axes[row, 0].set_ylim(60, 85)
    axes[row, 0].legend(fontsize=8)
    for bar, val in zip(bars, survival_by_alpha):
        axes[row, 0].text(
            bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Right: learning curves for all alpha values
    cmap   = plt.cm.Blues if algo == 'SARSA' else plt.cm.Oranges
    window = 1000
    for i, a_val in enumerate(alpha_values):
        c   = cmap(0.35 + i * 0.13)
        lw  = 2.5 if a_val == best_a else 1.0
        lab = f'alpha={a_val}' + (' ← best' if a_val == best_a else '')
        roll = pd.Series(alpha_grid_returns[(algo, a_val)]).rolling(window).mean()
        axes[row, 1].plot(roll, color=c, linewidth=lw, label=lab)

    axes[row, 1].axhline(pi_results['mean_return'], color='tomato',
                         linestyle='--', linewidth=1.5, label='PI ceiling')
    axes[row, 1].axhline(np.mean(rand_returns), color='gray',
                         linestyle=':', linewidth=1.5, label='Random baseline')
    axes[row, 1].set_xlabel('Episode')
    axes[row, 1].set_ylabel('Rolling mean return')
    axes[row, 1].set_title(
        f'{algo}: Learning Curves by Alpha ({N_OPT_EPISODES//1000}k episodes)',
        fontweight='bold')
    axes[row, 1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_alpha_grid_both.png', bbox_inches='tight')
plt.show()

# Auto-interpretation
print(f'Best alpha — SARSA: {best_alpha_sarsa}  |  Q-Learning: {best_alpha_ql}')
if best_alpha_sarsa < best_alpha_ql:
    print('SARSA prefers a smaller alpha — consistent with on-policy learning:')
    print('conservative updates help when the target includes exploratory actions.')
elif best_alpha_sarsa > best_alpha_ql:
    print('Interestingly SARSA prefers a larger alpha here — the sparse reward')
    print('structure may be dominating over the on/off-policy distinction.')
else:
    print('Both algorithms prefer the same alpha — suggesting the reward')
    print('structure of this environment drives the tuning more than algorithm type.')


### 7.2 — Stability Analysis: SARSA vs Q-Learning

We now evaluate the best configuration for each algorithm — best alpha, 200k episodes — and compare not just survival rate but **policy stability** (standard deviation of returns across evaluation episodes).

In a clinical setting this matters: an agent that saves 75% of patients reliably is preferable to one that averages 76% but occasionally fails badly. A lower standard deviation means more predictable treatment outcomes.


In [ ]:
# ── Stability evaluation using best configs ───────────────────────────────────
# We reuse the already-trained best models from the grid search above
# No additional training needed — just evaluate with more episodes

def evaluate_with_distribution(policy_array, n_episodes=2000, seed=SEED):
    """
    Evaluate a tabular policy over n_episodes and return full return distribution.
    Returns mean, std, survival rate, and raw returns array.
    """
    np.random.seed(seed)
    env_eval = make_sepsis_env()
    returns = []

    for _ in range(n_episodes):
        obs, _ = env_eval.reset(seed=np.random.randint(100_000))
        total_r, done = 0.0, False
        while not done:
            action = int(policy_array[int(obs)])
            obs, r, te, tr, _ = env_eval.step(action)
            total_r += r
            done = te or tr
        returns.append(total_r)

    env_eval.close()
    returns = np.array(returns)
    return {
        'mean_return'   : float(np.mean(returns)),
        'std_return'    : float(np.std(returns)),
        'survival_rate' : float(np.mean(returns > 0)) * 100,
        'returns_array' : returns,
    }


# Retrieve best Q-tables from grid search (already trained — no extra cost)
# Re-train only if the key doesn't exist (shouldn't happen)
print('Evaluating best SARSA configuration...')
sarsa_best_Q_tmp, _ = sarsa(
    n_episodes=N_OPT_EPISODES,
    alpha=best_alpha_sarsa,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
)
# Note: we retrain here because we didn't store Q_tables in the grid search
# To avoid this, you could store them during the grid search loop above
sarsa_best_policy = np.argmax(sarsa_best_Q_tmp, axis=1)
sarsa_best_dist   = evaluate_with_distribution(sarsa_best_policy, n_episodes=2000)
print(f'  Survival: {sarsa_best_dist["survival_rate"]:.1f}%  '
      f'Std: {sarsa_best_dist["std_return"]:.4f}')

print('Evaluating best Q-Learning configuration...')
ql_best_Q_tmp, _ = q_learning(
    n_episodes=N_OPT_EPISODES,
    alpha=best_alpha_ql,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
)
ql_best_policy = np.argmax(ql_best_Q_tmp, axis=1)
ql_best_dist   = evaluate_with_distribution(ql_best_policy, n_episodes=2000)
print(f'  Survival: {ql_best_dist["survival_rate"]:.1f}%  '
      f'Std: {ql_best_dist["std_return"]:.4f}')

# PI distribution for comparison
pi_dist = evaluate_with_distribution(pi_policy, n_episodes=2000)
print(f'  PI ceiling — Survival: {pi_dist["survival_rate"]:.1f}%  '
      f'Std: {pi_dist["std_return"]:.4f}')


In [ ]:
# ── Plot: stability and final comparison ──────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dist_configs = [
    ('Random Baseline',         {'returns_array': rand_returns,
                                 'mean_return': np.mean(rand_returns),
                                 'std_return': np.std(rand_returns),
                                 'survival_rate': survival_rate},  'gray'),
    ('Policy Iteration',        pi_dist,           'tomato'),
    (f'SARSA (best α={best_alpha_sarsa})',   sarsa_best_dist, 'steelblue'),
    (f'Q-Learning (best α={best_alpha_ql})', ql_best_dist,   'darkorange'),
]

# Left: return distribution overlaid histograms
for name, d, color in dist_configs:
    axes[0].hist(d['returns_array'], bins=30, alpha=0.45,
                 color=color, label=name, density=True, edgecolor='none')
axes[0].set_xlabel('Episode Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Return Distributions\n(2000 evaluation episodes)', fontweight='bold')
axes[0].legend(fontsize=8)

# Middle: mean ± std error bars
names  = [d[0].replace(' (', '\n(') for d in dist_configs]
means  = [d[1]['mean_return']  for d in dist_configs]
stds   = [d[1]['std_return']   for d in dist_configs]
colors = [d[2]                 for d in dist_configs]

axes[1].bar(range(len(names)), means, color=colors, alpha=0.8, edgecolor='white')
axes[1].errorbar(range(len(names)), means, yerr=stds,
                 fmt='none', color='black', capsize=5, linewidth=1.5)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, fontsize=8)
axes[1].set_ylabel('Mean Return ± Std')
axes[1].set_title('Mean Return with Variability\n(lower error bar = more reliable)',
                  fontweight='bold')
for i, (m, s) in enumerate(zip(means, stds)):
    axes[1].text(i, m + s + 0.005, f'σ={s:.3f}',
                 ha='center', fontsize=8)

# Right: original vs optimised survival rate
comp_labels = [
    'Random\nBaseline',
    'PI\nCeiling',
    'SARSA\nOriginal',
    f'SARSA\nBest (α={best_alpha_sarsa})',
    'Q-Lrn\nOriginal',
    f'Q-Lrn\nBest (α={best_alpha_ql})',
]
comp_values = [
    survival_rate,
    pi_results['survival_rate'],
    sarsa_results['survival_rate'],
    sarsa_best_dist['survival_rate'],
    ql_results['survival_rate'],
    ql_best_dist['survival_rate'],
]
comp_colors = ['gray', 'tomato',
               '#aed6f1', 'steelblue',
               '#fad7a0', 'darkorange']

bars = axes[2].bar(range(len(comp_labels)), comp_values,
                   color=comp_colors, alpha=0.9, edgecolor='white')
axes[2].set_xticks(range(len(comp_labels)))
axes[2].set_xticklabels(comp_labels, fontsize=8)
axes[2].set_ylabel('Survival Rate (%)')
axes[2].set_title('Survival Rate: Original vs Optimised\n(light = original, dark = optimised)',
                  fontweight='bold')
axes[2].set_ylim(60, 85)
for bar, val in zip(bars, comp_values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom',
                 fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_stability_comparison.png', bbox_inches='tight')
plt.show()


### 7.3 — Final Summary Table


In [ ]:
# ── Final Config A summary ────────────────────────────────────────────────────

final_table = pd.DataFrame([
    {'Algorithm': 'Random Baseline', 'Version': '—',
     'Episodes': '—', 'Alpha': '—',
     'Survival %': round(survival_rate, 1),
     'Mean Return': round(float(np.mean(rand_returns)), 4),
     'Std Return': round(float(np.std(rand_returns)), 4),
     'Gap to PI (pp)': round(pi_results['survival_rate'] - survival_rate, 1)},

    {'Algorithm': 'Policy Iteration', 'Version': 'Optimal (ceiling)',
     'Episodes': '—', 'Alpha': '—',
     'Survival %': round(pi_dist['survival_rate'], 1),
     'Mean Return': round(pi_dist['mean_return'], 4),
     'Std Return': round(pi_dist['std_return'], 4),
     'Gap to PI (pp)': 0.0},

    {'Algorithm': 'SARSA', 'Version': 'Original',
     'Episodes': '50k', 'Alpha': '0.3',
     'Survival %': round(sarsa_results['survival_rate'], 1),
     'Mean Return': round(sarsa_results['mean_return'], 4),
     'Std Return': round(float(np.std(sarsa_returns[-2000:])), 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - sarsa_results['survival_rate'], 1)},

    {'Algorithm': 'SARSA', 'Version': 'Optimised',
     'Episodes': f'{N_OPT_EPISODES//1000}k', 'Alpha': str(best_alpha_sarsa),
     'Survival %': round(sarsa_best_dist['survival_rate'], 1),
     'Mean Return': round(sarsa_best_dist['mean_return'], 4),
     'Std Return': round(sarsa_best_dist['std_return'], 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - sarsa_best_dist['survival_rate'], 1)},

    {'Algorithm': 'Q-Learning', 'Version': 'Original',
     'Episodes': '50k', 'Alpha': '0.3',
     'Survival %': round(ql_results['survival_rate'], 1),
     'Mean Return': round(ql_results['mean_return'], 4),
     'Std Return': round(float(np.std(ql_returns[-2000:])), 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - ql_results['survival_rate'], 1)},

    {'Algorithm': 'Q-Learning', 'Version': 'Optimised',
     'Episodes': f'{N_OPT_EPISODES//1000}k', 'Alpha': str(best_alpha_ql),
     'Survival %': round(ql_best_dist['survival_rate'], 1),
     'Mean Return': round(ql_best_dist['mean_return'], 4),
     'Std Return': round(ql_best_dist['std_return'], 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - ql_best_dist['survival_rate'], 1)},

    {'Algorithm': 'Double Q-Learning', 'Version': 'Original',
     'Episodes': '50k', 'Alpha': '0.3',
     'Survival %': round(dql_results['survival_rate'], 1),
     'Mean Return': round(dql_results['mean_return'], 4),
     'Std Return': round(float(np.std(dql_returns[-2000:])), 4),
     'Gap to PI (pp)': round(pi_dist['survival_rate'] - dql_results['survival_rate'], 1)},
])

display(final_table)
final_table.to_csv(f'{PLOTS_DIR}/configA_final_table.csv', index=False)

# Printed summary
print('=' * 72)
print('CONFIG A — FINAL SUMMARY')
print('=' * 72)
sarsa_gain = sarsa_best_dist['survival_rate'] - sarsa_results['survival_rate']
ql_gain    = ql_best_dist['survival_rate']    - ql_results['survival_rate']
print(f'SARSA improvement from optimisation      : +{sarsa_gain:.1f}pp')
print(f'Q-Learning improvement from optimisation : +{ql_gain:.1f}pp')
print()

sarsa_more_stable = sarsa_best_dist['std_return'] < ql_best_dist['std_return']
if sarsa_more_stable:
    print(f'SARSA is more stable (σ={sarsa_best_dist["std_return"]:.4f}) '
          f'than Q-Learning (σ={ql_best_dist["std_return"]:.4f})')
    print('In a clinical context, SARSA may be the preferred choice:')
    print('its on-policy conservatism produces more predictable treatment outcomes.')
else:
    print(f'Q-Learning is more stable (σ={ql_best_dist["std_return"]:.4f}) '
          f'than SARSA (σ={sarsa_best_dist["std_return"]:.4f})')
    print('Q-Learning dominates on both survival rate and consistency.')

print()
print('Remaining gap to PI ceiling after optimisation:')
print(f'  SARSA:      {pi_dist["survival_rate"] - sarsa_best_dist["survival_rate"]:.1f}pp')
print(f'  Q-Learning: {pi_dist["survival_rate"] - ql_best_dist["survival_rate"]:.1f}pp')
print()
print('Note: PI uses the full transition model (P, R). The remaining gap')
print('is the cost of model-free learning — not solvable with more episodes.')
print('Additionally, P is estimated from finite data, so PI optimises over')
print('a noisy model — it is the best achievable within Config A, not an')
print('absolute clinical upper bound.')
